# Домашнее задание блока 2. Лекции 3–4

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/homeworks/hw2/homework.ipynb)

Задание по лекциям 3 и 4: обучение без модели — Q-learning и SARSA, их цели обновления, on-policy и off-policy, Expected SARSA. Среды меняются от части к части, а алгоритмы — те же, что вы написали на семинаре 4: **возьмите `epsilon_greedy`, `sarsa_step`, `q_step` и `train` из своего решения семинара**, это и задумано.

Всего **100 баллов** и бонус:

| Часть | Среда | Что проверяем | Баллы |
|---|---|---|---|
| 1. Q-learning на озере 8×8 | знакомая карта из домашки 1 | без модели ценность известна только там, где агент был; какие настройки доводят $V$ до $V^*$ | 35 |
| 2. Такси по чужому логу | Taxi-v4, лог случайной стратегии | off-policy учится на чужих данных, on-policy — нет | 35 |
| 3. Expected SARSA на такси | Taxi-v4 | случайное $a'$ в цели SARSA: при большом шаге $\alpha$ ломает таблицу, среднее по стратегии — нет | 30 |
| ★ Бонус. Горная машина на сетке | MountainCar-v0 | таблица на непрерывном состоянии: когда сетка работает, а когда нет | +10 |

Каждое задание разбито на пронумерованные пункты, у каждого пункта — своя заготовка `TODO` в коде. Части с `assert` проверяются автоматически при запуске ячейки: если `assert` не упал, часть засчитана. Графики и письменные ответы проверяются вручную — пишите их прямо в markdown-ячейках вместо слова `TODO`.

Целиком ноутбук считается около двух с половиной минут (вместе с бонусной частью). Задание рассчитано на 2–3 часа.

**Не забудьте про квизы.** К этому блоку их два — по лекции 3 и по лекции 4. Пройдите оба в зачётном режиме и положите коды результатов в `quiz/week03.txt` и `quiz/week04.txt`.

**Дедлайн: 3 ноября 2026, 23:59 МСК.**

Первые две ячейки служебные: установка пакетов в Colab и подключение модулей с картинками. Запустите и идите дальше.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium[toy-text,classic-control]"], check=True)

In [ ]:
# Служебное: три модуля с картинками и обвязкой. Читать не нужно, сигнатуры — в docstring каждого.
import pathlib, sys, urllib.request

MODULES = {"lake_plots": "01-intro/seminar", "taxi_plots": "04-sarsa/seminar", "hw2_plots": "homeworks/hw2"}
for name, folder in MODULES.items():
    local = next((p for p in ["", f"../../{folder}", f"../../../{folder}", folder] if (pathlib.Path(p) / f"{name}.py").exists()), None)
    if local is None:                                              # в Colab скачиваем из репозитория курса
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/IlyaChichkanov/Reinforcement-learning/main/{folder}/{name}.py", f"{name}.py")
        local = ""
    sys.path.insert(0, str(pathlib.Path(local).resolve()))

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from IPython.display import display
from lake_plots import lake_desc, draw_policy, plot_values
from taxi_plots import make_taxi, draw_taxi, average_greedy_return
from hw2_plots import (make_lake, mdp_from_gym, success_rate_greedy, collect_random_log, animate_taxi,
                       plot_learning_curves, plot_vs, make_mountain_car, make_discretize, greedy_episode, plot_positions)

## Часть 1. Q-learning на озере 8×8 (35 баллов)

*Это раздел 2 семинара 4 на карте из домашки 1.* В домашке 1 точную $Q^*$ считала value iteration — у неё была модель $P$ и $R$. У Q-learning модели нет: он ходит по озеру и обновляет таблицу после каждого шага. Точный ответ из домашки 1 здесь служит меркой: насколько выученная таблица совпала с $Q^*$ и что для этого нужно.

### 1.1 Инструменты семинара (15 баллов)

Перенесите из своего решения семинара 4 четыре функции: `epsilon_greedy`, `sarsa_step`, `q_step` и `train`. В ячейке самопроверки — числовые проверки из лекции 4, а затем оба правила обучаются на обрыве и рисуются их кривые обучения. Должна получиться картинка из лекции 4: SARSA набирает при обучении больше, Q-learning — меньше, потому что ходит по краю.

In [ ]:
GAMMA = 0.95

def epsilon_greedy(Q, s, eps, rng):
    # TODO 1.1: из семинара 4 — с вероятностью eps случайное действие, иначе лучшее по строке Q[s] (ничьи — случайно)
    raise NotImplementedError

def sarsa_step(Q, s, a, r, s2, a2, done, alpha, gamma):
    # TODO 1.1: цель r + gamma * Q[s2, a2] (при done — просто r), затем сдвиг Q[s, a] к цели на долю alpha
    raise NotImplementedError

def q_step(Q, s, a, r, s2, a2, done, alpha, gamma):
    # TODO 1.1: то же, но в цели максимум строки Q[s2]; a2 не используется
    raise NotImplementedError

def train(env, step_fn, n_episodes=500, eps=0.1, alpha=0.5, gamma=1.0, seed=0,
          max_steps=500, eps_decay=1.0, eps_min=0.0):
    # TODO 1.1: цикл обучения из семинара 4 — возвращает (Q, массив return'ов по эпизодам)
    raise NotImplementedError

In [ ]:
# Самопроверка 1.1 — числа из лекции 4, затем кривые обучения на обрыве
rng = np.random.default_rng(0)
Q_demo = np.array([[0.10, -1.00, 0.20, 0.40]])
assert all(epsilon_greedy(Q_demo, 0, 0.0, rng) == 3 for _ in range(100)), "при eps = 0 выбор всегда жадный"
freq = np.mean([epsilon_greedy(Q_demo, 0, 0.2, rng) == 3 for _ in range(100_000)])
assert 0.84 < freq < 0.86, f"при eps = 0.2 лучшее действие выбирается с частотой 0.85, у вас {freq:.3f}"

Q = np.zeros((2, 4)); Q[0, 0] = 0.30; Q[1, 2] = 0.50; Q[1, 3] = 0.90
sarsa_step(Q, s=0, a=0, r=0.0, s2=1, a2=2, done=False, alpha=0.1, gamma=1.0)
assert np.isclose(Q[0, 0], 0.32), f"SARSA: 0.30 + 0.1 · (0.50 − 0.30) = 0.32, у вас {Q[0, 0]:.3f}"
Q[0, 0] = 0.30
q_step(Q, s=0, a=0, r=0.0, s2=1, a2=2, done=False, alpha=0.1, gamma=1.0)
assert np.isclose(Q[0, 0], 0.36), f"Q-learning: 0.30 + 0.1 · (0.90 − 0.30) = 0.36, у вас {Q[0, 0]:.3f}"

cliff = gym.make("CliffWalking-v1")
Q_sarsa, ret_sarsa = train(cliff, sarsa_step, seed=2)
Q_ql, ret_ql = train(cliff, q_step, seed=2)
assert Q_ql.shape == (48, 4) and len(ret_ql) == 500, "train возвращает таблицу 48 × 4 и return каждого эпизода"
print("OK: числа сходятся с лекцией; ниже — обрыв при ε = 0.1, сравните с картинкой из лекции 4")
plot_learning_curves({"SARSA": ret_sarsa, "Q-learning": ret_ql}, title="прогулка по обрыву, ε = 0.1, α = 0.5"); plt.show()

### 1.2 Q-learning против точного ответа (20 баллов)

`make_lake()` — то же озеро 8×8 без скольжения, что в домашке 1; готовая `mdp_from_gym` достаёт из среды модель $P[s, a, s']$, $R[s, a]$ в формате лекции 2. Клетки льда (нетерминальные) собраны в массив `cells`: по ним сравниваем выученную $V = \max_a Q$ с точной $V^* = \max_a Q^*$, мерка — наибольшая ошибка $\max_{s} |V(s) - V^*(s)|$ (готовая `max_error`).

1. Перенесите из домашки 1 `bellman_update` и `value_iteration` и получите $Q^*$, $V^*$. Терминальные клетки в модели ведут сами в себя с нулевой наградой — менять ничего не нужно.
2. Обучите Q-learning вашей функцией `train` с настройками семинара: $\varepsilon = 0.1$, $\alpha = 0.5$, $\gamma = 0.95$, 5000 эпизодов не длиннее 100 шагов, старт в углу. Нарисуйте в ряд три картинки: $V^*$, выученную $V$ (обе через `plot_values` с одним и тем же `vmax`) и жадную стратегию — ту, что в каждой клетке берёт действие с наибольшим $Q$ (`draw_policy`). Напечатайте наибольшую ошибку и число клеток льда, где $V$ осталась нулём.
3. Подберите настройки, при которых $V$ совпадает с $V^*$ **во всех** клетках льда: `max_error < 0.01`. Рычага три — $\varepsilon$, $\alpha$ и число эпизодов — плюс среда `make_lake(random_start=True)`, где каждый эпизод начинается в случайной клетке льда. Для трёх и более наборов настроек, включая набор из пункта 2, постройте кривые ошибки против числа эпизодов: бюджеты `BUDGETS`, одно обучение на бюджет, график `plot_vs`. Найденный набор запишите в словарь `BEST`, таблицу — в `Q_lake`.
4. Ответьте текстом: (а) почему при старте из угла часть клеток остаётся нулевой, хотя путь по таблице выучен, — что значит «без модели ценность известна только там, где агент был»; (б) какие рычаги понадобились и почему $\alpha = 1$ на этом озере не мешает, а на скользком озере помешал бы.

In [ ]:
lake = make_lake()
desc = lake_desc(lake)
P, R, terminal = mdp_from_gym(lake)
cells = np.flatnonzero(~terminal)                     # клетки льда: по ним сравниваем V с V*
BUDGETS = (250, 500, 1000, 2000, 5000)

def bellman_update(Q, P, R, gamma=GAMMA):
    # TODO 1.2 (1): как в домашке 1 (семинар 2)
    raise NotImplementedError

def value_iteration(P, R, gamma=GAMMA, tol=1e-10, max_iter=1000):
    # TODO 1.2 (1): как в домашке 1; вернуть Q* и число итераций
    raise NotImplementedError

Q_star, n_iter = value_iteration(P, R)
V_star = Q_star.max(axis=1)

def max_error(Q):
    """Наибольшая ошибка выученной V = max_a Q против V* по клеткам льда."""
    return np.abs(Q.max(axis=1) - V_star)[cells].max()

# TODO 1.2 (2): train(lake, q_step, n_episodes=5000, eps=0.1, alpha=0.5, gamma=GAMMA, max_steps=100, seed=0);
#               три картинки в ряд (V*, V, жадная стратегия); печать max_error и числа нулевых клеток льда

In [ ]:
# TODO 1.2 (3): для нескольких наборов настроек (eps, alpha, random_start) и каждого бюджета из BUDGETS обучить
#               Q-learning и посчитать max_error; кривые ошибки против бюджета — plot_vs;
#               лучший набор — в BEST = dict(eps=..., alpha=..., n_episodes=..., random_start=...), его таблица — в Q_lake
BEST, Q_lake = None, None

In [ ]:
# Самопроверка 1.2
assert Q_star.shape == (64, 4) and np.isclose(V_star[0], 0.95 ** 13), "V*(старт) = 0.95^13: кратчайший путь — 14 шагов"
assert np.allclose(Q_star, bellman_update(Q_star, P, R), atol=1e-6), "Q* должна удовлетворять уравнению оптимальности"
assert Q_lake is not None and Q_lake.shape == (64, 4) and isinstance(BEST, dict), "Q_lake — таблица 64 × 4, BEST — словарь настроек"
assert max_error(Q_lake) < 0.01, f"с настройками BEST V должна совпасть с V* во всех клетках льда, у вас max-ошибка {max_error(Q_lake):.3f}"
assert success_rate_greedy(make_lake(), Q_lake) == 1.0, "жадная стратегия по Q_lake должна доходить всегда"
print(f"OK: V совпала с V* (ошибка {max_error(Q_lake):.4f}), V(старт) = {Q_lake.max(axis=1)[0]:.4f} при точном {V_star[0]:.4f}")

_Ваш ответ:_ TODO

## Часть 2. Такси по чужому логу (35 баллов)

*Это раздел 5 лекции 4 на такси из семинара 4.* На лекции Q-learning по логу случайных блужданий восстановил ценности лучшей стратегии, а SARSA — ценности той стратегии, что собирала лог; мир там был маленький, и таблицы сверялись с точными. Здесь то же самое повторяется на такси.

**Среда — по чек-листу лекции.** Поле 5 × 5 с четырьмя остановками R, G, Y, B; пассажир ждёт на одной из них, его нужно отвезти на другую.
- *Состояние* — одно число от 0 до 499, в котором упакованы три вещи: где такси (25 клеток), где пассажир (5 вариантов: на одной из четырёх остановок или уже в машине) и куда ехать (4 остановки). Итого $25 \times 5 \times 4 = 500$ состояний — и 500 строк в таблице $Q$. Распаковать число обратно умеет `taxi.unwrapped.decode(s)`.
- *Действий 6*: вниз, вверх, вправо, влево, посадить пассажира, высадить.
- *Награды*: $-1$ за каждый шаг, $-10$ за посадку или высадку не на месте, $+20$ за доставку. Эпизод обрезается на 200 шагах.

**Мерка — проверка по таблице.** Такси ездит, в каждом состоянии выбирая действие с наибольшим $Q$, без случайных шагов; считается средняя сумма наград за поездку по 100 стартам — готовая `average_greedy_return`. Довозит такси пассажира (около $+8$) или нет (отрицательные числа) — видно сразу.

Готовая `collect_random_log(taxi, n_episodes, seed)` возвращает лог случайного таксиста: каждый переход записан пятёркой $(s, a, r, s', a')$ и флагом конца эпизода. Лог из $n$ эпизодов при одном `seed` — это начало лога из 3000.

1. Напишите `learn_from_log(log, step_fn, n_states, n_actions, alpha, gamma)`: таблица из нулей, один проход по логу, на каждом переходе одно обновление `step_fn`. Данные и цикл для обоих правил одни и те же, различается только `step_fn`.
2. Кривая обучения по данным: для логов из 250, 500, 1000, 2000 и 3000 эпизодов обучите по таблице с `q_step` и с `sarsa_step` при $\alpha = 0.1$ и постройте награду за поездку по таблице против размера лога (`plot_vs`). Таблицы по полному логу сохраните в `Q_log_ql` и `Q_log_sarsa`; напечатайте, сколько в полном логе переходов и успешных высадок (награда $+20$).
3. Подберите $\alpha$, при котором Q-learning довозит уже по логу из 500 эпизодов: напечатайте таблицу наград за поездку для $\alpha \in \{0.1, 0.5, 1.0\}$ и обоих правил; таблицу Q-learning при лучшем $\alpha$ сохраните в `Q_small`.
4. Анимация: проедьте по таблице `Q_log_ql` своими руками — цикл с `argmax` по строке таблицы и `env.step`, списки состояний и наград до конца эпизода (не больше 50 шагов), затем `display(animate_taxi(taxi, states, rewards))`. Напечатайте итоговую награду. Локально вместо GIF можно открыть окно pygame: `make_taxi(render_mode="human")` и `env.render()` после каждого шага.
5. Ответьте текстом: (а) почему по одному и тому же логу такси по таблице Q-learning довозит, а по таблице SARSA — нет: чья ценность записана в каждой из таблиц; (б) почему при $\alpha = 1$ Q-learning хватило лога в шесть раз короче и что случилось бы при $\alpha = 1$ в среде со случайными переходами.

In [ ]:
taxi = make_taxi()
N_STATES, N_ACTIONS = taxi.observation_space.n, taxi.action_space.n
LOG_SIZES = (250, 500, 1000, 2000, 3000)

s0, _ = taxi.reset(seed=3)
row, col, passenger, destination = taxi.unwrapped.decode(s0)
print(f"состояний {N_STATES}, действий {N_ACTIONS}; стартовое состояние {s0} = "
      f"(такси в ряду {row}, столбце {col}; пассажир на остановке {'RGYB'[passenger]}; ехать на {'RGYB'[destination]})")
draw_taxi(taxi, s0, title=f"состояние {s0}: жёлтая машина — такси, кружок — пассажир, звезда — цель"); plt.show()

def learn_from_log(log, step_fn, n_states, n_actions, alpha=0.1, gamma=1.0):
    # TODO 2 (1): таблица из нулей; один проход по логу — для каждой пятёрки (s, a, r, s2, a2, done) один step_fn
    raise NotImplementedError

# TODO 2 (2): для n в LOG_SIZES — лог collect_random_log(taxi, n_episodes=n, seed=0), две таблицы (q_step, sarsa_step,
#             alpha=0.1), награда за поездку по каждой (average_greedy_return); график plot_vs; таблицы по полному логу — Q_log_ql, Q_log_sarsa;
#             печать числа переходов и высадок в полном логе
Q_log_ql, Q_log_sarsa = None, None

In [ ]:
# TODO 2 (3): по логу из 500 эпизодов — таблица наград за поездку для alpha в (0.1, 0.5, 1.0) и обоих правил;
#             таблица Q-learning при лучшем alpha — в Q_small
Q_small = None

In [ ]:
# TODO 2 (4): поездка по таблице Q_log_ql своими руками — state, _ = taxi.reset(seed=3); пока эпизод не кончился
#             (не больше 50 шагов): action = argmax строки, шаг среды, копить states и rewards;
#             затем display(animate_taxi(taxi, states, rewards)) и печать итоговой награды

In [ ]:
# Самопроверка 2
assert Q_log_ql is not None and Q_log_ql.shape == (500, 6) and Q_log_sarsa is not None, "таблицы такси — 500 × 6"
_ql, _sarsa = average_greedy_return(taxi, Q_log_ql), average_greedy_return(taxi, Q_log_sarsa)
assert _ql > 5, f"Q-learning по логу из 3000 эпизодов должен довозить (эталон около +8), у вас {_ql:+.1f}"
assert _sarsa < 0, f"SARSA по тому же логу довозить не должен (эталон около −130), у вас {_sarsa:+.1f}"
assert Q_small is not None and average_greedy_return(taxi, Q_small) > 5, \
    "с подобранным α Q-learning должен довозить уже по логу из 500 эпизодов"
print(f"OK: один лог, два правила — {_ql:+.1f} против {_sarsa:+.1f}; по короткому логу {average_greedy_return(taxi, Q_small):+.1f}")

_Ваш ответ:_ TODO

## Часть 3. Expected SARSA на такси (30 баллов)

*Это раздел 7.1 лекции 4 на такси из семинара 4.* На семинаре SARSA с постоянным $\varepsilon = 0.1$ проваливал проверку по таблице, и помогало затухание $\varepsilon$. Есть второй способ — убрать из цели случайный выбор $a'$ и подставить среднее по стратегии:

$$
\text{цель} = r + \gamma \sum_{a'} \pi(a' \mid s')\, Q(s', a'),
\qquad
\pi(a' \mid s') = \frac{\varepsilon}{|A|} + (1 - \varepsilon)\,[a' = \arg\max Q(s', \cdot)].
$$

### 3.1 Правило обновления (12 баллов)

Напишите `make_expected_sarsa_step(eps)`: она возвращает функцию с той же сигнатурой, что `sarsa_step` и `q_step`, — тогда её можно отдать в ваш `train`. Вероятности $\pi(\cdot \mid s')$ считайте по формуле выше: $\varepsilon / |A|$ каждому действию и $1 - \varepsilon$ сверху лучшему.

In [ ]:
def make_expected_sarsa_step(eps):
    def expected_sarsa_step(Q, s, a, r, s2, a2, done, alpha, gamma):
        # TODO 3.1: вероятности ε-жадной стратегии в строке s2, цель r + gamma * (probs @ Q[s2]), сдвиг Q[s, a]
        raise NotImplementedError
    return expected_sarsa_step

In [ ]:
# Самопроверка 3.1 — та же ситуация, что в 1.1: строка s2 = [0, 0, 0.50, 0.90], eps = 0.2
Q = np.zeros((2, 4)); Q[0, 0] = 0.30; Q[1, 2] = 0.50; Q[1, 3] = 0.90
make_expected_sarsa_step(0.2)(Q, s=0, a=0, r=0.0, s2=1, a2=2, done=False, alpha=0.1, gamma=1.0)
assert np.isclose(Q[0, 0], 0.349), ("среднее по стратегии: 0.85 · 0.90 + 0.05 · 0.50 = 0.79, "
                                     f"сдвиг 0.30 + 0.1 · (0.79 − 0.30) = 0.349, у вас {Q[0, 0]:.3f}")
make_expected_sarsa_step(0.2)(Q, s=0, a=1, r=1.0, s2=1, a2=2, done=True, alpha=0.5, gamma=1.0)
assert np.isclose(Q[0, 1], 0.5), "после терминального перехода цель — просто r"
print("OK: Expected SARSA — между SARSA (0.32) и Q-learning (0.36) в той же ситуации: 0.349")

### 3.2 Большой шаг α: SARSA против Expected SARSA (18 баллов)

Шаг обучения $\alpha$ — доля цели, которую таблица переписывает в себя за одно обновление: при $\alpha = 0.1$ оценка сдвигается к цели на десятую часть, при $\alpha = 1$ цель записывается в таблицу целиком. Сравним, как два правила переносят большой шаг.

1. Обучите на такси `sarsa_step` и `make_expected_sarsa_step(0.1)` одной и той же функцией `train` при **постоянном** $\varepsilon = 0.1$, $\gamma = 1$, 5000 эпизодов по 200 шагов, сид 0 — для двух шагов обучения: $\alpha = 0.1$ и $\alpha = 1$. Для каждой из четырёх таблиц посчитайте награду за поездку (`average_greedy_return`), сложите в словарь `results[(имя, alpha)]` и напечатайте таблицей 2 × 2.
2. Нарисуйте кривые обучения обоих правил при $\alpha = 1$ (`plot_learning_curves`).
3. Ответьте текстом: (а) что в цели SARSA случайно и почему при $\alpha = 1$ это ломает таблицу, а при $\alpha = 0.1$ только мешает; почему у Expected SARSA такой зависимости нет; (б) чем этот способ отличается от затухания $\varepsilon$ из семинара — что у них общего и когда предпочесть какой.

In [ ]:
PARAMS = dict(n_episodes=5000, eps=0.1, gamma=1.0, max_steps=200, seed=0)
RULES = {"SARSA": sarsa_step, "Expected SARSA": make_expected_sarsa_step(0.1)}

# TODO 3.2 (1): для alpha в (0.1, 1.0) и обоих правил — train(taxi, step_fn, alpha=alpha, **PARAMS), награда за поездку
#               в results[(name, alpha)]; напечатать таблицу 2 × 2
results = None
# TODO 3.2 (2): кривые обучения обоих правил при alpha = 1 — plot_learning_curves

In [ ]:
# Самопроверка 3.2
assert results is not None and len(results) == 4, "results — четыре награды за поездку: два правила × два α"
assert results[("Expected SARSA", 1.0)] > 5, f"Expected SARSA при α = 1 должен довозить, у вас {results[('Expected SARSA', 1.0)]:+.1f}"
assert results[("SARSA", 1.0)] < 0, f"SARSA при α = 1 довозить не должен, у вас {results[('SARSA', 1.0)]:+.1f}"
print(f"OK: при α = 1 SARSA {results[('SARSA', 1.0)]:+.1f}, Expected SARSA {results[('Expected SARSA', 1.0)]:+.1f}")

_Ваш ответ:_ TODO

## ★ Бонус. Горная машина на сетке (+10 баллов)

*Новая среда — и первый шаг к блоку 3; баллы идут сверх 100.* В `MountainCar-v0` машина стоит в долине между двумя горами; мотор слабый, в лоб на правую гору не въехать, нужно раскачаться. Наблюдение — два числа: позиция $x \in [-1.2, 0.6]$ и скорость $v \in [-0.07, 0.07]$; действия — толкать влево, не толкать, толкать вправо; награда $-1$ за каждый шаг; эпизод кончается у флага ($x \ge 0.5$) или через 200 шагов.

Состояние непрерывное, таблица $Q$ по нему не строится. Старый приём — **разрезать на сетку**: $x$ и $v$ делим на `bins` одинаковых отрезков, состояние — пара номеров отрезков, таблица $Q$ размера `bins × bins × 3`. Готовая `make_discretize(car, bins)` возвращает функцию, которая переводит наблюдение в такую пару.

1. Каркас `train_on_grid` ниже повторяет ваш `train` для состояния-пары: `Q[s]` — строка из трёх чисел. Допишите обновление Q-learning (одна строка) и обучите сетку `bins = 20`: 5000 эпизодов, $\alpha = 0.1$, $\varepsilon = 0.1$, $\gamma = 0.99$ — около 30 секунд. Напечатайте среднюю длину эпизода по таблице по 20 стартам (`greedy_episode`; эпизод без финиша считается за 200 шагов).
2. Повторите для грубой сетки `bins = 6` и нарисуйте на одном графике позицию машины по шагам для эпизода по таблице обеих сеток (`plot_positions`). Таблицы сохраните в словарь `grids = {bins: Q}`.
3. Ответьте текстом: (а) почему сетка 6 не доезжает, а 20 доезжает — что теряется при слишком грубом разрезании; (б) что случилось бы с таблицей, будь состояние не двумя числами, а картинкой с экрана, и чем это лечат в блоке 3.

In [ ]:
car = make_mountain_car()

def train_on_grid(env, bins, n_episodes=5000, alpha=0.1, eps=0.1, gamma=0.99, seed=0):
    """Q-learning на сетке bins × bins; возвращает таблицу Q."""
    rng = np.random.default_rng(seed)
    discretize = make_discretize(env, bins)
    Q = np.zeros((bins, bins, env.action_space.n))
    for episode in range(n_episodes):
        obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
        s = discretize(obs)
        while True:
            a = int(rng.integers(env.action_space.n)) if rng.random() < eps else int(np.argmax(Q[s]))
            obs, r, terminated, truncated, _ = env.step(a)
            s2 = discretize(obs)
            # TODO ★ (1): одна строка — сдвиг Q[s][a] к цели r + gamma * max Q[s2] (при terminated будущего нет)
            raise NotImplementedError
            s = s2
            if terminated or truncated:
                break
    return Q

def mean_episode_length(env, Q, bins, n_starts=20):
    """Средняя длина эпизода по таблице (без случайных шагов) по n_starts стартам; без финиша — 200."""
    lengths = []
    for seed in range(n_starts):
        positions, ok = greedy_episode(env, Q, make_discretize(env, bins), seed=1000 + seed)
        lengths.append(len(positions) - 1 if ok else 200)
    return np.mean(lengths)

# TODO ★ (1): обучить сетку 20, напечатать среднюю длину эпизода по таблице
# TODO ★ (2): обучить сетку 6; позиции по шагам для обеих сеток на одном графике (greedy_episode + plot_positions);
#             таблицы — в словарь grids = {bins: Q}
grids = None

In [ ]:
# Самопроверка ★
assert grids is not None and {6, 20} <= set(grids) and grids[20].shape == (20, 20, 3), "grids — таблицы сеток 6 и 20"
_len = {bins: np.mean([len(greedy_episode(car, grids[bins], make_discretize(car, bins), seed=1000 + k)[0]) - 1
                       for k in range(20)]) for bins in (6, 20)}
assert _len[20] < 190, f"сетка 20 должна доезжать: средняя длина эпизода по таблице < 190, у вас {_len[20]:.0f}"
assert _len[6] > _len[20], f"сетка 6 хуже сетки 20: {_len[6]:.0f} против {_len[20]:.0f} шагов"
print(f"OK: средняя длина эпизода по таблице — сетка 6: {_len[6]:.0f}, сетка 20: {_len[20]:.0f}")

_Ваш ответ:_ TODO

## Чек-лист перед сдачей

- [ ] 1.1: `epsilon_greedy`, `sarsa_step`, `q_step`, `train` перенесены, самопроверка пройдена, кривые обрыва похожи на лекционные
- [ ] 1.2: `value_iteration` даёт $V^*(\text{старт}) = 0.95^{13}$; три картинки для старта в углу; кривые ошибки против бюджета; `BEST` и `Q_lake` проходят самопроверку; ответ
- [ ] 2: `learn_from_log`; график против размера лога; таблица по $\alpha$ и `Q_small`; анимация поездки по таблице с итоговой наградой; ответ
- [ ] 3.1: `make_expected_sarsa_step` проходит самопроверку
- [ ] 3.2: таблица 2 × 2 `results` и кривые обучения при $\alpha = 1$; ответ
- [ ] ★ (по желанию): сетки 20 и 6 обучены, длины эпизодов напечатаны, график позиций, ответ
- [ ] Коды квизов по лекциям 3 и 4 лежат в `quiz/week03.txt` и `quiz/week04.txt`
- [ ] Ноутбук выполнен целиком: «Restart & Run All» перед сдачей